# demo_judge — 대화 판단(judge) 모델: 데이터 · 파인튜닝 · 채점 · 응답 시간

**judge 의 과제**: 게임이 끝난 뒤 **한 판의 전체 채팅**(최대 10명 = 나 + 팀원 4 + 상대 5)을 **유저별로 모아** 읽고,
유저마다 **신고 권고**와 **고소장 작성 후보** 여부를 정한다. 1~5초면 되고, 5초 이상이면 화면에 로딩 창을 띄운다.

impl(문장 1개, 0.5초 미만)과 과제가 달라서 **모델도 따로** 둔다.

| | impl 모델 | judge 모델 |
|---|---|---|
| 입력 | 문장 1개 (최대 64토큰) | 유저 1명을 판단하기 위한 게임 1판 문서 (최대 512토큰) |
| 출력 | 유형 5개 점수 | `report`(신고 권고) · `complaint`(고소 후보) 점수 |
| 위치 | `models/analyze-classifier/` | `models/analyze-judge/` |

판단 순서: ① 메시지마다 impl 결과로 근거를 모은다 → ② **유저별 누적 점수와 흐름 규칙**(맞장구 가담, 여러 명의 몰이, 말리는 말 제외, 협박, 이어지는 욕설) → ③ (선택) **judge 모델**이 유저의 전체 채팅을 흐름과 함께 읽고 보완.

**결론부터:** 지금 데이터(학습 게임 36판)로는 judge 모델이 규칙보다 낫지 않다. 그래서 앱 기본값은 **규칙만**(`mode="rules"`)이고, 모델은 이 노트북에서 학습·비교까지만 한다(5장).

실행: 맨 위부터 전체 실행. `RETRAIN = False` 면 저장된 모델로 채점만 한다. 모든 대화는 **AI 가 쓴 가상 대화(검수 전)** 다.

## 0. 설정

In [1]:
import json
import os
import time
import warnings
from collections import Counter

from transformers.utils import logging as hf_logging

from clean_chat.core.config import resolve_repo_path
from clean_chat.modules.analyze import judge, judge_evaluate, judge_model, model, train

warnings.filterwarnings("ignore")
hf_logging.set_verbosity_error()
hf_logging.disable_progress_bar()
FIX = resolve_repo_path("fixtures/analyze")

BASE = "models/analyze-classifier"  # judge 모델의 출발점 = 파인튜닝한 impl 모델 (욕설·비난을 이미 안다). 문서가 길어 최대 512토큰
LR = 3e-5
BATCH_SIZE = 8  # 문서가 길어서 impl(16)보다 작게
EPOCHS = 30
PATIENCE = 10
VAL_RATIO = 0.15
SEED = 1
RETRAIN = True
OUT_DIR = "models/ft/demo-judge"  # git 제외. 앱이 쓰는 models/analyze-judge 는 건드리지 않는다

GAMES = FIX / "judge_games.jsonl"
print("judge 모델 라벨:", judge_model.LABELS, "· 최대 토큰", judge_model.MAX_LENGTH)
print("impl 모델(메시지별 근거용):", model.version() or "없음 → 규칙만")

judge 모델 라벨: ['report', 'complaint'] · 최대 토큰 512


impl 모델(메시지별 근거용): kcelectra-base-ft-0.3


## 1. 학습·채점에 쓴 데이터

게임 단위 대화 데이터 `judge_games.jsonl`. 한 줄이 게임 한 판이다: 명단 10명, 채팅(시각·채널·화자·원문·정규화), 그리고 **문제 있는 유저에게만** 붙인 정답.
`split=train` 게임은 규칙 조정과 judge 모델 학습에, `split=test` 게임은 채점에만 쓴다.

In [2]:
games = judge_evaluate.load_games(GAMES)
by_split = {s: [g for g in games if g["split"] == s] for s in ("train", "test")}
print(f"{'':<6}{'게임':>5}{'채팅 줄':>8}{'유저(나 제외)':>12}{'신고 권고':>9}{'고소 후보':>9}{'평화로운 판':>10}")
for s, gs in by_split.items():
    labels = [v for g in gs for v in g["labels"].values()]
    print(
        f"{s:<6}{len(gs):>6}{sum(len(g['messages']) for g in gs):>10}{9 * len(gs):>13}"
        f"{sum(bool(v.get('report')) for v in labels):>12}{sum(bool(v.get('complaint')) for v in labels):>11}"
        f"{sum(not g['labels'] for g in gs):>12}"
    )
lengths = sorted(len(g["messages"]) for g in games)
print(f"\n한 판의 채팅 줄 수: 최소 {lengths[0]} · 중앙값 {lengths[len(lengths) // 2]} · 최대 {lengths[-1]}")
print("정답 유형:", dict(Counter(t for g in games for v in g["labels"].values() for t in v.get("types", []))))

         게임    채팅 줄    유저(나 제외)    신고 권고    고소 후보    평화로운 판
train     36       683          324          37          8          10
test      18       384          162          20          6           3

한 판의 채팅 줄 수: 최소 16 · 중앙값 18 · 최대 32
정답 유형: {'blame': 50, 'profanity': 22, 'instigation': 7, 'family_insult': 4, 'sexual_harassment': 3}


### 게임 한 판 보기

킬뎃 언급이 **조롱**인 줄과 **정상 정보**인 줄이 한 판에 섞여 있다. 한 문장만 봐서는 구분하기 어렵다.

In [3]:
game = next(g for g in by_split["train"] if g["id"] == "A01")
print(game["id"], "—", game["note"])
print(
    "명단:",
    ", ".join(f"{p['name']}({p['champion']}·{'우리' if p['team'] == 'ORDER' else '상대'})" for p in game["players"]),
)
for m in game["messages"]:
    print(f"  {m['t']:>4}초 [{'팀' if m['channel'] == 'team' else '전체'}] {m['speaker']}: {m['text']}")
print("\n정답")
for name, v in game["labels"].items():
    print(
        f"  {name}: 신고 권고={v['report']} · 고소 후보={v['complaint']} · 나를 향함={v['toward_me']} · 대상={v['targets']}"
    )
    print(f"      근거: {v['reason']}")

A01 — 원딜이 정글의 킬뎃을 들먹이며 몰아가고 탑이 맞장구 — 같은 탑의 '상대 다리 4/0' 은 정상 정보
명단: 밤하늘별(가렌·우리), 초코우유(리 신·우리), 달빛고양이(제드·우리), 불꽃남자(징크스·우리), 나(쓰레쉬·우리), 얼음땡(다리우스·상대), 구름빵(비에고·상대), 은하수(아리·상대), 솜사탕(케이틀린·상대), 번개돌이(럭스·상대)
    60초 [팀] 밤하늘별: 다들 화이팅
    75초 [팀] 초코우유: 블루 스타트 리쉬 좀요
   140초 [팀] 나: 봇 2렙 먼저 찍어요
   260초 [팀] 불꽃남자: 정글 봇 한 번만 와줘
   300초 [팀] 초코우유: ㅇㅋ 바위게 먹고 갈게
   420초 [팀] 달빛고양이: 미드 미아 아리 위로 감
   470초 [팀] 초코우유: 아 갱 실패 ㅈㅅ
   480초 [팀] 불꽃남자: 리신 큐 다 빗나가네 ㅋㅋ
   650초 [팀] 밤하늘별: 상대 다리 4/0 이야 탑 오지 마
   700초 [팀] 초코우유: 용 싸움에서 또 짤림 미안
   710초 [팀] 불꽃남자: 정글 0/5/1 ㅋㅋㅋ
   715초 [팀] 밤하늘별: ㅇㅈ 정글 차이 심하네
   760초 [팀] 불꽃남자: 리신 킬 관여 몇이냐 진짜
   770초 [팀] 나: 그만하고 다음 용 준비하자
   900초 [팀] 초코우유: 나도 열심히 하는 중이야
   905초 [팀] 불꽃남자: 열심히 해서 0/5 면 접어야지 ㅋㅋ
   910초 [팀] 밤하늘별: ㄹㅇ ㅋㅋ
  1000초 [팀] 달빛고양이: 바론 시야 잡자
  1100초 [전체] 은하수: gg 수고
  1105초 [팀] 달빛고양이: ㅅㄱ

정답
  불꽃남자: 신고 권고=True · 고소 후보=False · 나를 향함=False · 대상=['초코우유']
      근거: 리신이 죽은 직후마다 '정글 0/5/1 ㅋㅋㅋ', '킬 관여 몇이냐', '0/5 면 접어야지' 로 킬뎃을 들먹이며 계속 조롱
  밤하늘별: 신고 권고=True · 고소 후보=False · 나를 향함=False · 대상=[

### 유저별로 모은 채팅

judge 는 채팅을 **유저별로 모아서** 본다. 같은 판에서 누구는 콜만 했고 누구는 조롱을 반복했다.

In [4]:
for name, lines in judge_evaluate.transcripts(game).items():
    gold = game["labels"].get(name, {})
    mark = "신고 권고" if gold.get("report") else "문제없음"
    print(f"{name} ({len(lines)}줄, 정답: {mark})")
    for line in lines:
        print("    ", line)

밤하늘별 (4줄, 정답: 신고 권고)
     다들 화이팅
     상대 다리 4/0 이야 탑 오지 마
     ㅇㅈ 정글 차이 심하네
     ㄹㅇ ㅋㅋ
초코우유 (5줄, 정답: 문제없음)
     블루 스타트 리쉬 좀요
     ㅇㅋ 바위게 먹고 갈게
     아 갱 실패 ㅈㅅ
     용 싸움에서 또 짤림 미안
     나도 열심히 하는 중이야
달빛고양이 (3줄, 정답: 문제없음)
     미드 미아 아리 위로 감
     바론 시야 잡자
     ㅅㄱ
불꽃남자 (5줄, 정답: 신고 권고)
     정글 봇 한 번만 와줘
     리신 큐 다 빗나가네 ㅋㅋ
     정글 0/5/1 ㅋㅋㅋ
     리신 킬 관여 몇이냐 진짜
     열심히 해서 0/5 면 접어야지 ㅋㅋ
얼음땡 (0줄, 정답: 문제없음)
구름빵 (0줄, 정답: 문제없음)
은하수 (1줄, 정답: 문제없음)
     gg 수고
솜사탕 (0줄, 정답: 문제없음)
번개돌이 (0줄, 정답: 문제없음)


## 2. 누적 점수와 흐름 규칙 (모델 없이)

메시지마다 impl 결과(등급·대상)를 근거로 모으고 유저별로 점수를 쌓는다.
- 특정인을 향한 악성 채팅 1건 = 1.0, 패드립·성희롱 = 3.0, 정치질이면 +0.5, 남의 악성 채팅에 맞장구 = 0.5
- 여러 명이 같은 사람을 공격하면 ×1.25. 점수는 한 판 안에서 거의 그대로 쌓인다(30분에 1/e 로 감쇠)
- **신고 권고**: 욕설·고강도 표현·맞장구 가담이 있거나 누적 1.5 이상 (욕설 없는 거친 말 한 번은 권고하지 않음)
- **고소 후보**: 나를 향한 패드립·성희롱, 또는 나를 향한 누적 점수 3.0 이상(욕설 없이 비난만이면 4.0 이상)
- 공격하던 사람에게 하는 "그만해라"·"선 넘지 마라" 는 근거에서 뺀다
- 흐름으로 이어지는 말: 나를 공격하던 사람의 **협박**("찾아간다", "어디 사는지")은 고강도로, 대상을 생략한 욕설("꺼져 그냥")은 같은 대상을 향한 것으로 쌓는다

In [5]:
req = judge_evaluate.to_request(game)
result = judge.judge(req, mode="rules")
print(f"{'유저':<8}{'전체 발화':>6}{'누적 점수':>8}{'나를 향한 점수':>10}  {'신고 권고':<6}{'고소 후보':<6} 대상")
for s in result.speakers:
    print(
        f"{s.speaker:<8}{s.total_messages:>8}{s.score:>10.2f}{s.score_toward_me:>12.2f}    {str(s.report):<8}{str(s.complaint_candidate):<8} {s.targets}"
    )
    for r in s.reasons:
        print("      ·", r)
    for e in s.evidence:
        print(f"        [{e.kind}] {e.text}")

유저       전체 발화   누적 점수  나를 향한 점수  신고 권고 고소 후보  대상
불꽃남자           5      3.54        0.00    True    False    ['초코우유']
      · 특정인을 향한 악성 채팅 3건
      · 여러 명이 같은 사람을 공격함 (대상: 초코우유)
        [direct] 리신 큐 다 빗나가네 ㅋㅋ
        [direct] 정글 0/5/1 ㅋㅋㅋ
        [direct] 리신 킬 관여 몇이냐 진짜
밤하늘별           4      1.25        0.00    True    False    ['초코우유']
      · 특정인을 향한 악성 채팅 1건
      · 여러 명이 같은 사람을 공격함 (대상: 초코우유)
        [direct] ㅇㅈ 정글 차이 심하네


## 3. judge 모델의 입력 문서

유저 한 명을 판단할 때마다 그 판의 대화를 문서 하나로 만든다. `▶` 가 판단 대상이 한 말, `나(...)` 가 내가 한 말, `·` 가 그 밖의 사람이다.
같은 판이라도 판단 대상이 바뀌면 표시가 바뀐다. 문서가 길면 대상의 말에서 먼 다른 사람의 말부터 뺀다.

In [6]:
target = next(iter(game["labels"]))
doc = judge_model.build_document(req.messages, req.players, req.my_name, target)
print(f"[판단 대상: {target}] 글자 수 {len(doc)}\n")
print(doc)
train_examples = judge_model.examples_from_games(by_split["train"])
test_examples = judge_model.examples_from_games(by_split["test"])
print(
    f"\n학습 문서 {len(train_examples)}개 (신고 {int(sum(y[0] for _, y in train_examples))} · 고소 {int(sum(y[1] for _, y in train_examples))})"
    f" · 시험 문서 {len(test_examples)}개 (신고 {int(sum(y[0] for _, y in test_examples))} · 고소 {int(sum(y[1] for _, y in test_examples))})"
)
print("말을 한 번도 안 한 유저는 문서를 만들지 않고 문제없음으로 본다.")

[판단 대상: 불꽃남자] 글자 수 457

나: 쓰레쉬 서폿 | 대상: 징크스 원딜 우리팀
· 가렌(우리): 다들 화이팅
· 리신(우리): 블루 스타트 리쉬 좀요
나(쓰레쉬): 봇 2렙 먼저 찍어요
▶ 징크스: 정글 봇 한 번만 와줘
· 리신(우리): ㅇㅋ 바위게 먹고 갈게
· 제드(우리): 미드 미아 아리 위로 감
· 리신(우리): 아 갱 실패 ㅈㅅ
▶ 징크스: 리신 큐 다 빗나가네 ㅋㅋ
· 가렌(우리): 상대 다리 4/0 이야 탑 오지 마
· 리신(우리): 용 싸움에서 또 짤림 미안
▶ 징크스: 정글 0/5/1 ㅋㅋㅋ
· 가렌(우리): ㅇㅈ 정글 차이 심하네
▶ 징크스: 리신 킬 관여 몇이냐 진짜
나(쓰레쉬): 그만하고 다음 용 준비하자
· 리신(우리): 나도 열심히 하는 중이야
▶ 징크스: 열심히 해서 0/5 면 접어야지 ㅋㅋ
· 가렌(우리): ㄹㅇ ㅋㅋ
· 제드(우리): 바론 시야 잡자
· 아리(상대/전체): gg 수고
· 제드(우리): ㅅㄱ

학습 문서 198개 (신고 37 · 고소 8) · 시험 문서 105개 (신고 20 · 고소 6)
말을 한 번도 안 한 유저는 문서를 만들지 않고 문제없음으로 본다.


## 4. 파인튜닝

impl 과 같은 학습 코드(`train.py`)를 쓰되 라벨이 `report`·`complaint` 이고 입력이 512토큰 문서다.
학습 문서가 200개 남짓이라 **양이 매우 적다** — 고소 후보 양성은 8개뿐이다.

미리 해 본 결과 학습이 불안정했다: 원래 `beomi/KcELECTRA-base` 에서 출발하면(15·30 epoch) 손실이 줄지 않았고, impl 모델에서 출발해도 학습률 5e-5 는 실패, 3e-5 만 배웠다. 아래는 그 배운 설정이다.

In [7]:
OUT = resolve_repo_path(OUT_DIR)
history = []
if RETRAIN:
    started = time.perf_counter()
    train.train(
        [GAMES],
        BASE,
        OUT,
        "demo-judge",
        epochs=EPOCHS,
        batch_size=BATCH_SIZE,
        lr=LR,
        seed=SEED,
        max_length=judge_model.MAX_LENGTH,
        val_ratio=VAL_RATIO,
        patience=PATIENCE,
        labels=judge_model.LABELS,
        examples=train_examples,
        on_epoch=history.append,
        log=lambda line: print(line.replace(str(OUT), OUT_DIR)),
    )
    print(f"학습 시간 {time.perf_counter() - started:.0f}초")
meta = json.loads((OUT / "meta.json").read_text(encoding="utf-8"))
print(
    {k: meta[k] for k in ("base", "train_rows", "val_rows", "best_epoch", "val_macro_f1", "lr", "seed", "max_length")}
)
print("검증 F1:", meta["val_f1"])

학습 169문장 · 검증 29문장 · mps · 최대 30 epoch · lr 3e-05 · base=models/analyze-classifier


  epoch 1/30  loss 1.0068  검증 F1 평균 0.152


  epoch 2/30  loss 1.0027  검증 F1 평균 0.077


  epoch 3/30  loss 1.0119  검증 F1 평균 0.056


  epoch 4/30  loss 0.9485  검증 F1 평균 0.154


  epoch 5/30  loss 0.8026  검증 F1 평균 0.208


... (14 outputs left out)

  epoch 20/30  loss 0.0104  검증 F1 평균 0.364


  epoch 21/30  loss 0.0096  검증 F1 평균 0.364


  epoch 22/30  loss 0.0090  검증 F1 평균 0.364  → 조기 종료
  가장 좋은 epoch 12 (검증 F1 평균 0.833) 의 가중치를 저장


저장: models/ft/demo-judge  (평가: uv run python -m clean_chat.modules.analyze.evaluate <CSV> --split test)
학습 시간 665초
{'base': 'models/analyze-classifier', 'train_rows': 169, 'val_rows': 29, 'best_epoch': 12, 'val_macro_f1': 0.8333, 'lr': 3e-05, 'seed': 1, 'max_length': 512}
검증 F1: {'report': 0.6667, 'complaint': 1.0}


In [8]:
if history:
    top = max(h["loss"] for h in history)
    print(f"{'epoch':>5}  {'학습률':>9}  {'손실':>7}  {'':<20}  {'검증F1':>6}")
    for h in history:
        print(
            f"{h['epoch']:>5}  {h['lr']:>10.2e}  {h['loss']:>8.4f}  {'█' * round(20 * h['loss'] / top):<20}  {h.get('val_f1', 0):>7.3f}  {'●' * round(20 * h.get('val_f1', 0))}"
        )

epoch        학습률       손실                          검증F1
    1    1.00e-05    1.0068  ████████████████████    0.152  ●●●
    2    2.00e-05    1.0027  ████████████████████    0.077  ●●
    3    3.00e-05    1.0119  ████████████████████    0.056  ●
    4    2.89e-05    0.9485  ███████████████████     0.154  ●●●
    5    2.78e-05    0.8026  ████████████████        0.208  ●●●●
    6    2.67e-05    0.5861  ████████████            0.786  ●●●●●●●●●●●●●●●●
    7    2.56e-05    0.4159  ████████                0.308  ●●●●●●
    8    2.44e-05    0.3171  ██████                  0.444  ●●●●●●●●●
    9    2.33e-05    0.4601  █████████               0.222  ●●●●
   10    2.22e-05    0.3306  ███████                 0.325  ●●●●●●
   11    2.11e-05    0.1978  ████                    0.235  ●●●●●
   12    2.00e-05    0.1342  ███                     0.833  ●●●●●●●●●●●●●●●●●
   13    1.89e-05    0.0935  ██                      0.250  ●●●●●
   14    1.78e-05    0.0882  ██                      0.333  ●●●●●●●
  

## 5. 채점 — 정답률 · 오답률

**시험 게임**(학습에 쓰지 않은 판)에서 유저 단위로 채점한다. 세 가지 방식을 나란히 본다.
- `rules`: 누적 점수와 흐름 규칙만 (**앱 기본값**) · `model`: judge 모델만 · `hybrid`: 둘 중 하나라도 그렇다고 하면
- 정답률 = 맞힌 유저 / 전체 유저(나 제외). 대부분의 유저는 문제가 없으므로 정답률은 높게 나온다 → **재현율(놓친 비율)과 정밀도(잘못 잡은 비율)** 를 같이 봐야 한다.

In [9]:
os.environ[judge_model.ENV_VAR] = str(OUT)
model._load.cache_clear()
judge.judge(req, mode="hybrid")  # 모델 로딩을 속도 측정에서 뺀다


def pct(v):
    return "   -  " if v is None else f"{v * 100:5.1f}%"


scores = {}
for split in ("test", "train"):
    print(
        f"\n■ {split} 게임 {len(by_split[split])}판 · 유저 {9 * len(by_split[split])}명"
        + ("  ← 채점용" if split == "test" else "  ← 학습에 쓴 판 (참고)")
    )
    print(
        f"{'방식':<8}{'판단':<8}{'정답률':>7}{'오답률':>8}{'정밀도':>8}{'재현율':>8}{'F1':>7}{'정답 수':>6}{'오탐':>5}{'놓침':>5}"
    )
    for mode in ("rules", "model", "hybrid"):
        res = judge_evaluate.evaluate(by_split[split], lambda r, mode=mode: judge.judge(r, mode=mode))
        scores[split, mode] = res
        for key, label in (("report", "신고 권고"), ("complaint", "고소 후보")):
            m = res[key]
            print(
                f"{mode:<8}{label:<8}{pct(m['accuracy']):>9}{pct(1 - m['accuracy']):>9}{pct(m['precision']):>9}{pct(m['recall']):>9}"
                f"{pct(m['f1']):>9}{m['support']:>6}{m['fp']:>6}{m['fn']:>6}"
            )


■ test 게임 18판 · 유저 162명  ← 채점용
방식      판단          정답률     오답률     정밀도     재현율     F1  정답 수   오탐   놓침


rules   신고 권고       97.5%     2.5%    94.4%    85.0%    89.5%    20     1     3
rules   고소 후보       99.4%     0.6%   100.0%    83.3%    90.9%     6     0     1


model   신고 권고       96.3%     3.7%    76.9%   100.0%    87.0%    20     6     0
model   고소 후보       95.7%     4.3%    42.9%    50.0%    46.2%     6     4     3


hybrid  신고 권고       95.7%     4.3%    74.1%   100.0%    85.1%    20     7     0
hybrid  고소 후보       96.9%     3.1%    55.6%    83.3%    66.7%     6     4     1

■ train 게임 36판 · 유저 324명  ← 학습에 쓴 판 (참고)
방식      판단          정답률     오답률     정밀도     재현율     F1  정답 수   오탐   놓침


rules   신고 권고       98.8%     1.2%    90.2%   100.0%    94.9%    37     4     0
rules   고소 후보      100.0%     0.0%   100.0%   100.0%   100.0%     8     0     0


model   신고 권고       98.1%     1.9%    89.7%    94.6%    92.1%    37     4     2
model   고소 후보      100.0%     0.0%   100.0%   100.0%   100.0%     8     0     0


hybrid  신고 권고       97.8%     2.2%    84.1%   100.0%    91.4%    37     7     0
hybrid  고소 후보      100.0%     0.0%   100.0%   100.0%   100.0%     8     0     0


In [10]:
res = scores["test", "rules"]
print(f"시험 게임에서 rules(기본값)가 틀린 유저 {len(res['errors'])}건")
for e in res["errors"]:
    kind = "놓침" if e["gold"] else "오탐"
    print(
        f"  {e['game']} {e['user']} [{'신고 권고' if e['what'] == 'report' else '고소 후보'} {kind}] 누적 점수 {e['score']}"
    )
    if e["reason"]:
        print(f"      정답 근거: {e['reason']}")

시험 게임에서 rules(기본값)가 틀린 유저 5건
  B17 은하수 [신고 권고 놓침] 누적 점수 1.25
      정답 근거: 상대의 비꼼에 435초 'ㄹㅇ 쟤 상대 편 맞음', 665초 'ㅇㅈ 천사 정글', 1005초 전체 채팅 '인정합니다 가져가세요' 로 같은 팀 정글 몰이에 가담함
  B17 번개돌이 [신고 권고 놓침] 누적 점수 1.25
      정답 근거: 욕설은 없지만 240·430·660·1000초에 전체 채팅으로 우리 정글만 골라 '기부 감사', '여섯 번째 선수', '천사세요', 'mvp 드릴게요' 로 네 번 비꼼
  C12 파란우산 [신고 권고 놓침] 누적 점수 1.0
      정답 근거: 문장만 보면 킬뎃 사실이지만 진이 죽은 직후마다 '0/1 ㅋ'(312초), '0/3 ㅋㅋ'(520초), '0/4/0 대단하다'(800초), '딜량 봐라'(1100초)로 반복 조롱
  C16 은하수 [신고 권고 오탐] 누적 점수 1.81
  C18 민들레 [고소 후보 놓침] 누적 점수 1.85
      정답 근거: 애매: 욕설은 없지만 '명사수다'(260초), '덫 위치 예술'(450초), '캐리 감사합니다'(640초), '우리 진 최고지'(652초), '역시 우리 원딜'(900초), '많이 배웁니다'(1200초)로 그만하라는데도 나를 여섯 번 비꼬아 반복 모욕으로 봄. 욕설·패드립이 없어 고소 후보에서 뺄 수도 있음


### 한 판의 판단 결과

시험 게임 하나를 골라 유저별 판단(규칙)과, 참고로 judge 모델 점수(`mode="hybrid"`)를 함께 본다.

In [11]:
sample = next(g for g in by_split["test"] if any(v.get("complaint") for v in g["labels"].values()))
out = judge.judge(judge_evaluate.to_request(sample), mode="hybrid")
print(sample["id"], "—", sample["note"], f"({len(sample['messages'])}줄)\n")
print(f"{'유저':<8}{'발화':>4}{'누적':>7}{'나 향함':>7}  {'신고':<6}{'고소':<6} 모델 점수")
for s in out.speakers:
    print(
        f"{s.speaker:<8}{s.total_messages:>5}{s.score:>8.2f}{s.score_toward_me:>8.2f}   {str(s.report):<7}{str(s.complaint_candidate):<7} {s.model_scores}"
    )
    for r in s.reasons:
        print("      ·", r)
print("\n정답:", {k: {"신고": v["report"], "고소": v["complaint"]} for k, v in sample["labels"].items()})

B11 — 우리 원딜이 나(서폿)에게 욕설을 판 내내 반복(우회 표기) — 고소 후보. 나는 사과만 함 (19줄)

유저        발화     누적   나 향함  신고    고소     모델 점수
번개돌이        8    5.63    5.63   True   True    {'report': 0.9902, 'complaint': 0.6951}
      · 특정인을 향한 악성 채팅 8건
      · 나를 향한 표현 8건 (누적 점수 5.63)
      · 나를 향한 악성 채팅이 반복됨 (누적 5.63) → 고소 후보
얼음땡         2    0.00    0.00   True   False   {'report': 0.9895, 'complaint': 0.3316}
      · 대화 흐름 모델이 신고 권고로 판단 (점수 0.99) — 한 문장씩으로는 잡히지 않은 표현
솜사탕         2    1.00    0.00   False  False   {'report': 0.0093, 'complaint': 0.0057}
      · 특정인을 향한 악성 채팅 1건
      · 욕설 없는 거친 말 한 번 (누적 1.0 < 1.5) → 신고 권고 안 함

정답: {'번개돌이': {'신고': True, '고소': True}}


## 6. 응답 시간

기준: **1판 1~5초**(5초 이상이면 로딩 창). 한 판을 판단하는 시간 = 메시지마다 impl 실행 + 유저별 문서를 judge 모델에 통과.

In [12]:
print(f"{'방식':<8}{'평균':>7}{'중앙값':>8}{'p95':>7}{'최대':>7}  1초 이상  5초 이상(로딩 창)")
for mode in ("rules", "model", "hybrid"):
    lat = scores["test", mode]["latency_sec"]
    print(
        f"{mode:<8}{lat['mean']:>8.2f}초{lat['median']:>7.2f}초{lat['p95']:>6.2f}초{lat['max']:>6.2f}초{lat['over_fast']:>8}판{lat['over_slow']:>10}판"
    )

model._load.cache_clear()
started = time.perf_counter()
judge.judge(req)
print(f"\n모델 첫 로딩 + 첫 판 (rules): {time.perf_counter() - started:.2f}초 (impl 모델을 불러온다, 한 번만)")
longest = max(games, key=lambda g: len(g["messages"]))
for mode in ("rules", "hybrid"):
    judge.judge(judge_evaluate.to_request(longest), mode=mode)
    started = time.perf_counter()
    judge.judge(judge_evaluate.to_request(longest), mode=mode)
    print(f"가장 긴 판({len(longest['messages'])}줄) {mode}: {time.perf_counter() - started:.2f}초")

방식           평균     중앙값    p95     최대  1초 이상  5초 이상(로딩 창)
rules       0.36초   0.34초  0.51초  0.53초       0판         0판
model       0.85초   0.83초  1.22초  1.25초       5판         0판
hybrid      0.84초   0.79초  1.22초  1.27초       4판         0판



모델 첫 로딩 + 첫 판 (rules): 0.49초 (impl 모델을 불러온다, 한 번만)


가장 긴 판(32줄) rules: 0.52초


가장 긴 판(32줄) hybrid: 1.03초


## 정리와 한계

- judge 는 **유저별로 모은 전체 채팅**을 본다. 지금은 누적 점수와 흐름 규칙이 기본이다. judge 전용 모델은 따로 학습해 봤지만 지금 데이터로는 규칙보다 낫지 않아 기본값에서 뺐다. 규칙이 못 잡는 것(욕설 없는 비꼼의 반복, 킬뎃만 던지는 조롱)이 바로 모델이 맡아야 할 부분이라, 게임 데이터가 늘면 다시 비교할 만하다.
- 규칙의 일부(협박, 이어지는 욕설)는 **시험 게임의 오답을 본 뒤** 넣었다. 그래서 시험 수치는 순수한 시험 성적이 아니고 낙관적이다.
- **데이터가 매우 적다.** 학습 게임 36판(문서 약 200개), 시험 게임 18판이고 고소 후보 정답은 시험에 6명뿐이다. 유저 1명의 차이가 재현율 17%p 다. 위 수치는 방향만 보여 준다.
- 대화는 모두 AI 가 썼다(세 묶음을 따로 작성). 실제 채팅은 더 지저분하고 더 길다.
- 고소 후보를 모델만으로 정한 경우에는 근거 문장과 판례 요건 점검이 붙지 않는다(사람이 확인해야 한다). **법률 자문이 아니다.**
- judge 는 아직 앱 흐름에 연결되어 있지 않다(계약 변경 필요 — README "다음 단계").
- 응답 시간은 이 노트북을 돌린 PC(CPU) 기준이다. Windows PC 에서는 따로 재야 한다.